# Mini-ColBERTv2 on SciFact (Colab GPU)

Run the full pipeline: encode → k-means → 2-bit residual → IVF → search → CAD-MaxSim → evaluate.

In [ ]:
# 1. Clone repo
!git clone https://github.com/HoangNguyen689/colbert-hoangnd.git
%cd colbert-hoangnd

In [ ]:
# 2. Install dependencies (pin transformers<5 to stay compatible with colbert-ai)
!pip install -q "transformers==4.57.6" "colbert-ai==0.2.22" scikit-learn datasets beir tqdm numpy ninja

In [ ]:
# 3. Ensure ninja is on PATH for colbert-ai C++ extension compilation
import os, sys
os.environ['PATH'] += ':' + os.path.join(sys.prefix, 'bin')
!ninja --version

In [ ]:
# 4. Encode SciFact corpus and queries (uses official colbert-ir/colbertv2.0 checkpoint)
!python -m mini_colbert.encode --data_root data --output_dir outputs --doc_batch_size 64 --query_batch_size 64

In [ ]:
# 5. Verify encode succeeded
import os
required = ['outputs/doc_embeddings.pt', 'outputs/metadata.pt', 'outputs/query_embeddings.pt', 'outputs/idf.pt', 'outputs/qrels_test.json']
missing = [p for p in required if not os.path.exists(p)]
assert not missing, f'Missing files: {missing}'
print('All required files generated.')

In [ ]:
# 6. Build k-means centroids (K=8192)
!python -m mini_colbert.build_index --emb_path outputs/doc_embeddings.pt --output_dir outputs --K 8192 --n_samples 200000

In [ ]:
# 7. 2-bit residual quantization
!python -m mini_colbert.compress_residuals --emb_path outputs/doc_embeddings.pt --centroids_path outputs/centroids.pt --codes_path outputs/codes.pt --output_dir outputs --nbits 2

In [ ]:
# 8. Build IVF
!python -m mini_colbert.ivf --emb_path outputs/doc_embeddings.pt --codes_path outputs/codes.pt --output_dir outputs

In [ ]:
# 9. Approximate search + MaxSim
!python -m mini_colbert.search --index_dir outputs --query_emb outputs/query_embeddings.pt --output outputs/search_results.pt --ncells 4 --topk 100 --device cuda

In [ ]:
# 10. CAD-MaxSim 4 ablations (rerank same top-100 Original candidates)
!python -m mini_colbert.cad_maxsim --index_dir outputs --query_emb outputs/query_embeddings.pt --idf outputs/idf.pt --qrels outputs/qrels_test.json --output_dir outputs --ncells 4 --topn 100 --topk 100 --device cuda

In [ ]:
# 11. Evaluate all variants
import os
for name in ['search_results.pt', 'rankings_original.json', 'rankings_idf.json', 'rankings_adaptive.json', 'rankings_cad.json']:
    print(f'=== {name} ===')
    path = f'outputs/{name}'
    if os.path.exists(path):
        !python -m mini_colbert.evaluate --rankings {path} --qrels outputs/qrels_test.json

In [ ]:
# 12. Compare with official baseline
!python -m mini_colbert.compare_official --rankings outputs/rankings_original.json --qrels outputs/qrels_test.json